In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class Head(nn.Module):

    def __init__(self, head_size, d_model, block_size):
        super().__init__()
        self.key   = nn.Linear(d_model, head_size, bias=False)
        self.query = nn.Linear(d_model, head_size, bias=False)
        self.value = nn.Linear(d_model, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.head_size = head_size

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        scores = q @ k.transpose(-2, -1) * (self.head_size ** -0.5)
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        out = weights @ v
        return out

In [3]:
B = 4
T = 8
d_model = 32
head_size = 16
block_size = 8

head = Head(head_size=head_size, d_model=d_model, block_size=block_size)
x = torch.randn(B, T, d_model)
out = head(x)

print(f"Input  shape: {x.shape}")
print(f"Output shape: {out.shape}")

Input  shape: torch.Size([4, 8, 32])
Output shape: torch.Size([4, 8, 16])


In [4]:
with torch.no_grad():
    k = head.key(x)
    q = head.query(x)
    scores = q @ k.transpose(-2, -1) * (head_size ** -0.5)
    scores = scores.masked_fill(head.tril[:T, :T] == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)

print("Attention weights for batch item 0:")
print(weights[0].round(decimals=2))
print(f"\nDoes each row sum to 1? {weights[0].sum(dim=-1).round(decimals=4)}")

Attention weights for batch item 0:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2800, 0.7200, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4700, 0.1200, 0.4200, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3500, 0.1200, 0.2900, 0.2400, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1700, 0.1400, 0.2500, 0.2200, 0.2200, 0.0000, 0.0000, 0.0000],
        [0.1800, 0.1400, 0.2000, 0.1500, 0.1700, 0.1700, 0.0000, 0.0000],
        [0.1600, 0.1200, 0.1500, 0.1400, 0.1400, 0.1400, 0.1400, 0.0000],
        [0.1400, 0.2200, 0.1300, 0.1200, 0.1000, 0.1200, 0.1000, 0.0700]])

Does each row sum to 1? tensor([1., 1., 1., 1., 1., 1., 1., 1.])


In [7]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size, d_model, block_size):
        super().__init__()
        self.heads = nn.ModuleList([
            Head(head_size, d_model, block_size) for _ in range(num_heads)
        ])
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out


B = 4
T = 8
d_model = 32
num_heads = 4
head_size = d_model // num_heads   # = 8
block_size = 8

mha = MultiHeadAttention(num_heads=num_heads, head_size=head_size,
                          d_model=d_model, block_size=block_size)

x = torch.randn(B, T, d_model)
out = mha(x)

print(f"Input  shape: {x.shape}")
print(f"Output shape: {out.shape}")
print(f"head_size per head: {head_size}")
print(f"num_heads × head_size = {num_heads} × {head_size} = {num_heads * head_size}")

Input  shape: torch.Size([4, 8, 32])
Output shape: torch.Size([4, 8, 32])
head_size per head: 8
num_heads × head_size = 4 × 8 = 32


In [11]:
import sys
sys.path.append('..')
from src.models.transformer import TransformerLanguageModel

vocab_size = 65
d_model    = 32
num_heads  = 4
num_layers = 3
block_size = 8

model = TransformerLanguageModel(vocab_size, d_model, num_heads, num_layers, block_size)
print("Model created successfully")

Model created successfully


In [12]:
x = torch.randint(0, vocab_size, (4, 8))
y = torch.randint(0, vocab_size, (4, 8))

logits, loss = model(x, y)
print(f"x shape:      {x.shape}")
print(f"logits shape: {logits.shape}")
print(f"loss:         {loss.item():.4f}")

x shape:      torch.Size([4, 8])
logits shape: torch.Size([32, 65])
loss:         4.3081


In [13]:
total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total:,}")

Total parameters: 42,369


In [14]:
import math
baseline = math.log(vocab_size)
print(f"Theoretical baseline loss: {baseline:.4f}")
print(f"Actual initial loss:        {loss.item():.4f}")
print(f"Difference: {abs(loss.item() - baseline):.4f}")


Theoretical baseline loss: 4.1744
Actual initial loss:        4.3081
Difference: 0.1337
